# Chapter 18 — SQL Project: Step 1## Supervised Fine-Tuning on Spider Dataset[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**T4 GPU recommended. Runtime: ~60-90 minutes.**This is Part 1 of the 5-part SQL fine-tuning project from Chapter 18. Run the 5 notebooks in order:1. `18_01_sft_sql.ipynb` ← **you are here**2. `18_02_reward_model_sql.ipynb`3. `18_03_ppo_sql.ipynb`4. `18_04_evaluation.ipynb`5. `18_05_dpo_comparison.ipynb`

In [ ]:
# Setup
import torch
print(f'GPU: {torch.cuda.is_available()}')
try:
    from datasets import load_dataset
    spider = load_dataset('spider', trust_remote_code=True)
    print(f'Spider loaded: {len(spider["train"])} train, {len(spider["validation"])} val')
except Exception as e:
    print(f'Dataset: {e}')

## Full code from Chapter 18, Section 18.2See Chapter 18 in the book for the complete annotated code. This notebook runs the exact code from section 18.2.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig
import torch

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Replace with Mistral-7B for better results

def format_sft_example(example):
    prompt = f'You are an expert SQL developer. Generate correct, efficient SQL.\n\nDatabase: {example["db_id"]}\nQuestion: {example["question"]}\n\nSQL Query:'
    return {'prompt': prompt, 'response': example['query'],
            'text': prompt + ' ' + example['query'] + '<EOS>'}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Uncomment to run full training (requires T4+ GPU and ~60min):
# train_data = spider['train'].map(format_sft_example)
# val_data = spider['validation'].map(format_sft_example)
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto')
# lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','v_proj'], task_type=TaskType.CAUSAL_LM)
# model = get_peft_model(model, lora_config)
# trainer = SFTTrainer(model=model, args=SFTConfig(output_dir='./sql_sft', num_train_epochs=3, ...), ...)
# trainer.train()
print('SFT notebook ready. Uncomment training code and run on GPU.')

## Expected Results After SFT| Metric | Base Model | After SFT ||--------|------------|----------|| Exact Match | ~48% | ~68% || Execution Accuracy | ~52% | ~73% || Syntax Valid | ~78% | ~94% |Proceed to `18_02_reward_model_sql.ipynb` for the reward model training step.